In [1]:
import os
import time
import base64
import threading
import fitz  # PyMuPDF
from docx import Document as DocxDocument
from pptx import Presentation
import ollama

from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

C:\Users\dines\AppData\Local\Temp\ipykernel_8240\3913515238.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings


In [2]:
DATA_DIR = "E:\Personal\Projects\Hacktoberfest2026\Week1"
DB_DIR = "./chroma_db"
LLM_MODEL = "llama3.1:8b"
OCR_MODEL = "glm-ocr"
EMBEDDING_MODEL = "nomic-embed-text"

os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
def call_glm_ocr(image_bytes: bytes) -> str:
    """Sends image bytes to local GLM-OCR via Ollama to extract tables/text."""
    try:
        img_b64 = base64.b64encode(image_bytes).decode("utf-8")
        response = ollama.generate(
            model=OCR_MODEL,
            prompt="Analyze this image. Extract all text. If there is a table, format it perfectly in Markdown.",
            images=[img_b64]
        )
        return response.get("response", "")
    except Exception as e:
        print(f"\n[!] OCR Agent Error: {e}")
        return ""

In [4]:
def process_pdf(file_path: str) -> str:
    doc = fitz.open(file_path)
    full_text = ""
    
    for page_num, page in enumerate(doc):
        print(f"  -> Processing PDF page {page_num + 1}/{len(doc)}")
        page_text = page.get_text().strip()
        
        # CASE 1: Scanned Page (Very little to no selectable text)
        # We render the entire page as a high-res image and send it to the OCR model
        if len(page_text) < 50: 
            print("     Detected scanned page or complex layout, running full-page OCR...")
            # dpi=150 gives a good balance of OCR readability and speed
            pix = page.get_pixmap(dpi=150) 
            img_bytes = pix.tobytes("png")
            ocr_text = call_glm_ocr(img_bytes)
            if ocr_text:
                full_text += f"\n[OCR Page {page_num + 1}]:\n{ocr_text}\n"
                
        # CASE 2: Standard PDF (Has text, but might contain embedded charts/tables)
        else:
            full_text += page_text + "\n"
            
            # Process embedded images safely
            image_list = page.get_images(full_info=True)
            for img_index, img_info in enumerate(image_list):
                xref = img_info[0]
                try:
                    # Extract the image as a Pixmap
                    pix = fitz.Pixmap(doc, xref)
                    
                    # Force conversion to standard RGB if it's CMYK or something weird
                    # Ollama vision models require standard image formats
                    if pix.n - pix.alpha >= 4: 
                        pix = fitz.Pixmap(fitz.csRGB, pix)
                    
                    img_bytes = pix.tobytes("png")
                    
                    print(f"     Sending image {img_index + 1} from page {page_num + 1} to OCR...")
                    ocr_text = call_glm_ocr(img_bytes)
                    if ocr_text:
                        full_text += f"\n[Extracted Visual/Table from Page {page_num + 1}]:\n{ocr_text}\n"
                except Exception as e:
                    print(f"     [!] Failed to process image on page {page_num + 1}: {e}")
                    
    return full_text

In [5]:
def process_docx(file_path: str) -> str:
    doc = DocxDocument(file_path)
    full_text = "\n".join([p.text for p in doc.paragraphs])
    
    for rel in doc.part.rels.values():
        if "image" in rel.target_ref:
            img_bytes = rel.target_part.blob
            ocr_text = call_glm_ocr(img_bytes)
            if ocr_text:
                full_text += f"\n[Extracted Visual/Table Content]:\n{ocr_text}\n"
    return full_text

In [6]:
def process_pptx(file_path: str) -> str:
    prs = Presentation(file_path)
    full_text = ""
    for slide in prs.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                full_text += shape.text + "\n"
            if shape.shape_type == 13:  # PICTURE
                img_bytes = shape.image.blob
                ocr_text = call_glm_ocr(img_bytes)
                if ocr_text:
                    full_text += f"\n[Extracted Visual/Table Content]:\n{ocr_text}\n"
    return full_text

In [7]:
def process_file(file_path: str):
    print(f"Agent reading: {file_path}...")
    ext = os.path.splitext(file_path)[-1].lower()
    text = ""
    
    if ext in [".txt", ".md"]:
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
    elif ext == ".pdf":
        text = process_pdf(file_path)
    elif ext == ".docx":
        text = process_docx(file_path)
    elif ext == ".pptx":
        text = process_pptx(file_path)
    else:
        print(f"Skipping unsupported file: {ext}")
        return []

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_text(text)
    return [Document(page_content=c, metadata={"source": file_path}) for c in chunks]

In [8]:
class IngestionAgent(FileSystemEventHandler):
    def __init__(self, vectorstore):
        self.vectorstore = vectorstore

    def on_created(self, event):
        if event.is_directory:
            return
        
        file_path = event.src_path
        print(f"\n[+] Agent detected new file drop: {file_path}")
        time.sleep(2) 
        
        docs = process_file(file_path)
        if docs:
            self.vectorstore.add_documents(docs)
            print(f"[+] Agent successfully embedded {len(docs)} chunks into ChromaDB.\n>>> ", end="", flush=True)

In [ ]:
def main():
    print("Initializing Offline Agentic RAG System...")
    
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    vectorstore = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    
    event_handler = IngestionAgent(vectorstore)
    observer = Observer()
    observer.schedule(event_handler, DATA_DIR, recursive=False)
    observer.start()
    print(f"Agent is now watching the '{DATA_DIR}' folder for new files.")

    llm = ChatOllama(model=LLM_MODEL, temperature=0.2)
    prompt = ChatPromptTemplate.from_template("""
    You are an intelligent offline assistant. Use the following extracted context to answer the user's question. 
    The context may contain text, or Markdown tables/data extracted from images. 
    
    Context:
    {context}
    
    Question: {input}
    
    Answer clearly and concisely based ONLY on the context provided.
    """)
    
    document_chain = create_stuff_documents_chain(llm, prompt)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    qa_chain = create_retrieval_chain(retriever, document_chain)

    print("\nSystem ready! Drop files into the 'data' folder to update the DB automatically.")
    print("Type your questions below (or 'exit' to quit):\n")
    
    try:
        while True:
            user_input = input(">>> ")
            if user_input.lower() in ["exit", "quit"]:
                break
            if not user_input.strip():
                continue
            
            print("Thinking...")
            response = qa_chain.invoke({"input": user_input})
            print(f"\nLlama 3.1: {response['answer']}\n")
            
    except KeyboardInterrupt:
        pass
    finally:
        observer.stop()
        observer.join()
        print("\nSystem shut down securely.")

if __name__ == "__main__":
    main()

Initializing Offline Agentic RAG System...
Agent is now watching the 'E:\Personal\Projects\Hacktoberfest2026\Week1' folder for new files.

System ready! Drop files into the 'data' folder to update the DB automatically.
Type your questions below (or 'exit' to quit):



>>>  at what time I should sleep?


Thinking...

Llama 3.1: You should try to sleep at 10:30 PM.

